# Fine Tuning Large Language Model - Model

In this workshop, you will learn how to fine tune the prompts and the LLMs to enhance and improves its response.

In [20]:
!uv pip install evaluate
!uv pip install -U torchao

Using Python 3.12.13 environment at: /usr
Checked 1 package in 140ms
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 104ms
Prepared 1 package in 129ms
Uninstalled 1 package in 57ms
Installed 1 package in 16ms
 - torchao==0.10.0
 + torchao==0.17.0


In [1]:
# Import libraries
import torch, time
import pandas as pd
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Trainer, GenerationConfig, TrainingArguments

from peft import PeftModel, LoraConfig, get_peft_model, TaskType

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1745: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


In [2]:
# TODO: Load and explore the following datasets

dataset_name = "knkarthick/dialogsum"
model_name = "google/flan-t5-base"

dataset = load_dataset(dataset_name)

print(dataset)

print(dataset.shape)


DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})
{'train': (12460, 4), 'validation': (500, 4), 'test': (1500, 4)}


In [6]:
# TODO: Print a record
idx = 100
for k, v in dataset['train'][idx].items():
   print(f"{k}: {v}")


id: train_100
dialogue: #Person1#: I have a problem with my cable.
#Person2#: What about it?
#Person1#: My cable has been out for the past week or so.
#Person2#: The cable is down right now. I am very sorry.
#Person1#: When will it be working again?
#Person2#: It should be back on in the next couple of days.
#Person1#: Do I still have to pay for the cable?
#Person2#: We're going to give you a credit while the cable is down.
#Person1#: So, I don't have to pay for it?
#Person2#: No, not until your cable comes back on.
#Person1#: Okay, thanks for everything.
#Person2#: You're welcome, and I apologize for the inconvenience.
summary: #Person1# has a problem with the cable. #Person2# promises it should work again and #Person1# doesn't have to pay while it's down.
topic: cable


## Fine tuning the LLM model

In this workshop we will be turning the <code>google/flan-t5-base</code> model.

In [3]:
# Utility function to dump a model's tunable parameters

def print_trainable_model_params(model):
   trainable_model_params = 0
   all_model_params = 0
   for _, param in model.named_parameters():
      all_model_params += param.numel()
      if param.requires_grad:
         trainable_model_params += param.numel()
   return f"Trainable parameters: {trainable_model_params}\nTotal parameters: {all_model_params}\nPercentable of trainable parameters: {100 * trainable_model_params / all_model_params:.2f}%"

In [4]:
# TODO: Load model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [5]:
# TODO: Print number of trainable parameters
print(print_trainable_model_params(model))

Trainable parameters: 247577856
Total parameters: 247577856
Percentable of trainable parameters: 100.00%


### Preprocess the dialogue dataset

We will train the model to summarize dialogue by creating a dialogue-summary pair for the LLM to process. The dialogue is the training data and the summary is the label. This is supervized learning.

The prompt will be as follows

```
Summarize the following dialogue.\n
\n
Fred: ...\n
Barney: ...\n
\n
Summary:\n
Summary of the conversation between Fred and Barney
```

The prompt and the summary will be tokenized for the LLM

In [6]:
# Utitlity function to prepare the data for training
# Tokenize function
def tokenize_fn(data):
   start_prompt = 'Summarize the following dialogue.\n\n'
   end_prompt = '\n\nSummary:'
   prompt = [ start_prompt + d + end_prompt for d in data['dialogue'] ]
   summary = data['summary']
   data['input_ids'] = tokenizer(prompt, padding="max_length", truncation=True, return_tensors="pt").input_ids
   data['labels'] = tokenizer(summary, padding="max_length", truncation=True, return_tensors="pt").input_ids
   return data


In [7]:
# TODO: prepare the data for training with the tokenize_fn function
tokenized_dataset = dataset.map(tokenize_fn, batched=True)



In [9]:
# TODO: Verify prepared data
idx = 100
for k, v in tokenized_dataset['train'][idx].items():
   print(f"{k}: {v}")


id: train_100
dialogue: #Person1#: I have a problem with my cable.
#Person2#: What about it?
#Person1#: My cable has been out for the past week or so.
#Person2#: The cable is down right now. I am very sorry.
#Person1#: When will it be working again?
#Person2#: It should be back on in the next couple of days.
#Person1#: Do I still have to pay for the cable?
#Person2#: We're going to give you a credit while the cable is down.
#Person1#: So, I don't have to pay for it?
#Person2#: No, not until your cable comes back on.
#Person1#: Okay, thanks for everything.
#Person2#: You're welcome, and I apologize for the inconvenience.
summary: #Person1# has a problem with the cable. #Person2# promises it should work again and #Person1# doesn't have to pay while it's down.
topic: cable
input_ids: [12198, 1635, 1737, 8, 826, 7478, 5, 1713, 345, 13515, 536, 4663, 10, 27, 43, 3, 9, 682, 28, 82, 4807, 5, 1713, 345, 13515, 357, 4663, 10, 363, 81, 34, 58, 1713, 345, 13515, 536, 4663, 10, 499, 4807, 65, 118,

In [10]:
idx = 10
dec_input = tokenizer.decode(tokenized_dataset['train'][idx]['input_ids'])
dec_labels = tokenizer.decode(tokenized_dataset['train'][idx]['labels'])

print(dec_input)
print(dec_labels)

Summarize the following dialogue. #Person1#: Could you do me a favor? #Person2#: Sure. What is it? #Person1#: Could you run over to the store? We need a few things. #Person2#: All right. What do you want me to get? #Person1#: Well, could you pick up some sugar? #Person2#: Okay. How much? #Person1#: A small bag. I guess we also need a few oranges. #Person2#: How many? #Person1#: Oh, let's see. . . About six. #Person2#: Anything else? #Person1#: Yes. We're out of milk. #Person2#: Okay. How much do you want me to get? A gallon? #Person1#: No. I think a half gallon will be enough. #Person2#: Is that all? #Person1#: I think so. Have you got all that? #Person2#: Yes. That's small bag of sugar, four oranges, and a half gallon of milk. #Person1#: Do you have enough money? #Person2#: I think so. #Person1#: Thanks very much. I appreciate it. Summary:</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad

In [11]:
# TODO: Remove id, dialogue, summary and topic columns from dataset. We only want input_ids and labels
tokenized_dataset = tokenized_dataset.remove_columns(['id', 'dialogue', 'summary', 'topic'])
print(tokenized_dataset['train'][idx])

{'input_ids': [12198, 1635, 1737, 8, 826, 7478, 5, 1713, 345, 13515, 536, 4663, 10, 9348, 25, 103, 140, 3, 9, 4971, 58, 1713, 345, 13515, 357, 4663, 10, 10625, 5, 363, 19, 34, 58, 1713, 345, 13515, 536, 4663, 10, 9348, 25, 661, 147, 12, 8, 1078, 58, 101, 174, 3, 9, 360, 378, 5, 1713, 345, 13515, 357, 4663, 10, 432, 269, 5, 363, 103, 25, 241, 140, 12, 129, 58, 1713, 345, 13515, 536, 4663, 10, 1548, 6, 228, 25, 1432, 95, 128, 2656, 58, 1713, 345, 13515, 357, 4663, 10, 16036, 5, 571, 231, 58, 1713, 345, 13515, 536, 4663, 10, 71, 422, 2182, 5, 27, 3382, 62, 92, 174, 3, 9, 360, 5470, 7, 5, 1713, 345, 13515, 357, 4663, 10, 571, 186, 58, 1713, 345, 13515, 536, 4663, 10, 3359, 6, 752, 31, 7, 217, 5, 3, 5, 3, 5, 4504, 1296, 5, 1713, 345, 13515, 357, 4663, 10, 21345, 1307, 58, 1713, 345, 13515, 536, 4663, 10, 2163, 5, 101, 31, 60, 91, 13, 3702, 5, 1713, 345, 13515, 357, 4663, 10, 16036, 5, 571, 231, 103, 25, 241, 140, 12, 129, 58, 71, 12486, 106, 58, 1713, 345, 13515, 536, 4663, 10, 465, 5, 27, 

In [ ]:
# TODO: Verify dataset again


### Tune model with pre-processed dataset

We will use [<code>Trainer</code>](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#api-reference%20][%20transformers.Trainer) to train the original model. The training result will be written out. The trainer will be configure with [<code>TrainingArgument</code>](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.TrainingArguments)

In [15]:
# CUDA information

print('CUDA available: ', torch.cuda.is_available())
if torch.cuda.is_available():
   print('B16 supported: ', torch.cuda.is_bf16_supported())
   torch.cuda.set_device(0)
   print('Current device: ', torch.cuda.current_device())
   print('CUDA device name: ', torch.cuda.get_device_name(0))

CUDA available:  False


## Fine tuning the LLM Model with Low-Rank Adaptation (LoRA) / Parameter Efficient Fine Tuning (PEFT)

We will add a LoRA adapter to the LLM (flan-t5-base) and train the adapter. The original LLM will be frozen. The adapter can be combined with the original LLM during inferencing.

In [16]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [12]:
# TODO: Configure LoRA
rank = 8
lora_config = LoraConfig(
    r=rank,
    lora_alpha=rank,
    target_modules=['q', 'k'],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)


In [13]:
# TODO: Add LoRA to the LLM model to be trained
# Load our base model google/flan-t5-base
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
lora_model = get_peft_model(base_model, lora_config)


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [14]:
print(lora_model)

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): T5ForConditionalGeneration(
      (shared): Embedding(32128, 768)
      (encoder): T5Stack(
        (embed_tokens): Embedding(32128, 768)
        (block): ModuleList(
          (0): T5Block(
            (layer): ModuleList(
              (0): T5LayerSelfAttention(
                (SelfAttention): T5Attention(
                  (q): lora.Linear(
                    (base_layer): Linear(in_features=768, out_features=768, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=768, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=768, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
              

In [15]:
# TODO: Print number of parameters, compare LoRA to the original model
print(print_trainable_model_params(lora_model))

Trainable parameters: 884736
Total parameters: 248462592
Percentable of trainable parameters: 0.36%


In [16]:
# TODO: Train model with LoRA
lora_trainer_args = TrainingArguments(
  output_dir="./checkpoints",
  auto_find_batch_size=True,
  learning_rate=1e-3, # 0.0001
  num_train_epochs=1
)


In [ ]:
# TODO: Create trainer and train model
trainer = Trainer(
    model = lora_model,
    args = lora_trainer_args,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['test']
)

# Start the training....
# trainer.train()

# my_model_name = "chukmunnlee/my-dialogsum-t5-base-instruct"
# #save the trained model
# lora_model.save_pretrained(my_model_name)
# # save the tokenizer
# tokenizer.save_pretrained(my_model_name)


### Use a trained LoRA model

The training will take a few hours and over many iterations.

For the purpose of this workshop we use a save model [intotheverse/peft-dialogue-summary-checkpoint](https://huggingface.co/intotheverse/peft-dialogue-summary-checkpoint).

In [18]:
#TODO: Load the original model and add the pre-trained LoRA adaptation to the model
peft_dialogue_summary_checkpoint = 'intotheverse/peft-dialogue-summary-checkpoint'

original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

lora_model = PeftModel.from_pretrained(original_model, peft_dialogue_summary_checkpoint, is_trainable=False)



Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


adapter_config.json:   0%|          | 0.00/334 [00:00<?, ?B/s]

adapter_model.bin:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

In [19]:
original_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [21]:
idx = 200
dialogue = dataset['test'][idx]['dialogue']
summary = dataset['test'][idx]['summary']
print(dialogue)
print('-----------------')
print(summary)

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.
-----------------
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.


In [22]:
prompt = f"""
Summarize the following dialogue.

{dialogue}

Summary:
"""
print(prompt)


Summarize the following dialogue.

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

Summary:



In [25]:
enc_prompt = tokenizer(prompt, return_tensors='pt')

enc_summary = lora_model.generate(**enc_prompt)

generated_summary = tokenizer.decode(enc_summary[0], skip_special_tokens=True)
print(generated_summary)
print('========================')
print(summary)

#Person1# recommends adding a painting program to #Person2#'s
#Person1# teaches #Person2# how to upgrade software and hardware in #Person2#'s system.


## Evaluate LoRA model

In [ ]:
# TODO: Evaluate LoRA model against the original



In [26]:
# Prepare data for evaluation
dialogues = []
summaries = []
orig_model_summaries = []
lora_model_summaries = []
config = GenerationConfig(max_new_tokens=200)

for i in range(5):
   print(f'i = {i}')
   d = dataset['test'][i]['dialogue']
   s = dataset['test'][i]['summary']
   prompt = f"Summarize the following conversation.\n\n{d}\n\nSummary:"
   tokenized_prompt = tokenizer(prompt, return_tensors='pt').input_ids
   orig_resp = original_model.generate(input_ids=tokenized_prompt, generation_config=config)
   orig_resp_text = tokenizer.decode(orig_resp[0], skip_special_tokens=True)
   lora_resp = lora_model.generate(input_ids=tokenized_prompt, generation_config=config)
   lora_resp_text = tokenizer.decode(lora_resp[0], skip_special_tokens=True)

   summaries.append(s)
   orig_model_summaries.append(orig_resp_text)
   lora_model_summaries.append(lora_resp_text)

zipped_summaries = list(zip(summaries, orig_model_summaries, lora_model_summaries))
df = pd.DataFrame(zipped_summaries, columns=['label', 'original_model_summary', 'lora_model_summary'])
df

i = 0
i = 1
i = 2
i = 3
i = 4


,label,original_model_summary,lora_model_summary
0,Ms. Dawson helps #Person1# to write a memo to ...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
1,In order to prevent employees from wasting tim...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
2,Ms. Dawson takes a dictation for #Person1# abo...,The memo is to be distributed to all employees...,#Person1# asks Ms. Dawson to take a dictation ...
3,#Person2# arrives late because of traffic jam....,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic and #Person1# s...
4,#Person2# decides to follow #Person1#'s sugges...,The traffic jam at the Carrefour intersection ...,#Person2# got stuck in traffic and #Person1# s...


### Evaluate models with ROUGE/Bleu metrics

Recall-Oriented Understudy for Gisting Evaluate ([ROUGE](https://pub.aimind.so/unveiling-the-power-of-rouge-metrics-in-nlp-b6d3f96d3363)) is a set of metrics used to evaluate the quality of machine-generated text, such as summaries and translations. ROUGE metrics compare the generated text to a human-written reference and measure the overlap between the two.

The metrics range between 0 and 1, with higher scores indicating higher similarity between the baseline and generated text.

In [28]:
!uv pip install rouge_score

Using Python 3.12.13 environment at: /usr
Resolved 9 packages in 1.28s
Prepared 1 package in 353ms
Installed 1 package in 1ms
 + rouge-score==0.1.2


In [29]:
# TODO: create ROUGE metrics
rouge = evaluate.load('rouge')


In [30]:
# TODO: Evaluate the original model's result
orig_model_result = rouge.compute(
    references=summaries,
    predictions=orig_model_summaries,
    use_aggregator=True,
    use_stemmer=True,
)
lora_model_result = rouge.compute(
    references=summaries,
    predictions=lora_model_summaries,
    use_aggregator=True,
    use_stemmer=True,
)

print('Original Model:')
print(orig_model_result)
print('---------------------')
print('Lora Model:')
print(lora_model_result)

Original Model:
{'rouge1': np.float64(0.17391941391941393), 'rouge2': np.float64(0.03820816864295125), 'rougeL': np.float64(0.13362637362637358), 'rougeLsum': np.float64(0.13364468864468865)}
---------------------
Lora Model:
{'rouge1': np.float64(0.3397321482608658), 'rouge2': np.float64(0.1024924201890494), 'rougeL': np.float64(0.27097500453912726), 'rougeLsum': np.float64(0.27286086061853176)}


In [ ]:
# TODO: Evaluate with Bleu metrics
